# Glyph Dual Replay Rank-32 LoRA Trainer

Lossless Glyph8 sampling replay with dual adapters: `glyph_ledger` and `sampling_ledger`.

In [ ]:
# Optional Kaggle dependency install cell. Run only if packages are missing.
# !pip install -q transformers peft accelerate bitsandbytes datasets


In [ ]:
from pathlib import Path
import shutil, os, json

WORK_DIR = Path('/kaggle/working/glyph_dual_replay')
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(WORK_DIR)


In [ ]:
# Copy the implementation file into Kaggle working dir if it is attached as a dataset.
# Update SOURCE_DIR if needed.
SOURCE_DIRS = [Path('/kaggle/input/glyph-dual-replay'), Path('/kaggle/input'), Path('.')]
for sd in SOURCE_DIRS:
    cand = list(sd.rglob('glyph_dual_replay_lora.py')) if sd.exists() else []
    if cand:
        shutil.copy2(cand[0], WORK_DIR / 'glyph_dual_replay_lora.py')
        print('copied', cand[0])
        break
else:
    raise FileNotFoundError('Attach glyph_dual_replay_lora.py as a Kaggle dataset or upload it to the notebook.')


In [ ]:
!python /kaggle/working/glyph_dual_replay/glyph_dual_replay_lora.py selftest --tmp-dir /kaggle/working/glyph_dual_replay/selftest


In [ ]:
# Set these paths for your run.
MODEL_PATH = '/kaggle/input/YOUR_MODEL'
INPUT_DATA = '/kaggle/input/YOUR_DATA/train.jsonl'
LEDGER_DIR = '/kaggle/working/glyph_dual_replay/ledger'
TRAIN_JSONL = '/kaggle/working/glyph_dual_replay/dual_train.jsonl'
ADAPTER_OUT = '/kaggle/working/glyph_dual_replay/adapters'


In [ ]:
!python /kaggle/working/glyph_dual_replay/glyph_dual_replay_lora.py ingest --input "$INPUT_DATA" --out-dir "$LEDGER_DIR"
!python /kaggle/working/glyph_dual_replay/glyph_dual_replay_lora.py build --ledger-dir "$LEDGER_DIR" --out-jsonl "$TRAIN_JSONL" --max-records 50000 --seed 918
!python /kaggle/working/glyph_dual_replay/glyph_dual_replay_lora.py verify --ledger-dir "$LEDGER_DIR" --train-jsonl "$TRAIN_JSONL"


In [ ]:
!python /kaggle/working/glyph_dual_replay/glyph_dual_replay_lora.py train \
  --model "$MODEL_PATH" \
  --train-jsonl "$TRAIN_JSONL" \
  --out-dir "$ADAPTER_OUT" \
  --rank 32 \
  --alpha 64 \
  --dropout 0.05 \
  --load-in-4bit \
  --gradient-checkpointing \
  --batch-size 1 \
  --grad-accum 8 \
  --max-length 2048 \
  --glyph-warmup-steps 100


In [ ]:
!python /kaggle/working/glyph_dual_replay/glyph_dual_replay_lora.py export --ledger-dir "$LEDGER_DIR" --out-zip /kaggle/working/glyph_dual_replay/glyph_dual_replay_ledgers.zip
!find /kaggle/working/glyph_dual_replay -maxdepth 3 -type f | sort | sed -n '1,80p'
